# Data Pipeline — Interactive Notebook

This notebook mirrors the data-processing pipeline from `code/data_cleaning/`. Each stage loads, cleans, and displays the resulting DataFrames so you can scroll through the data interactively.

## Setup

Load all required packages.

In [ ]:
using DataFrames
using CSV
using Arrow
using Statistics
using Printf

### Paths

Adjust `PROJECT_ROOT` if your notebook lives somewhere other than `code/notebooks/`.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────
# Assumes this notebook is in:  project_root/code/notebooks/
const PROJECT_ROOT      = joinpath(@__DIR__, "..", "..")
const DATA_DIR          = joinpath(PROJECT_ROOT, "data")
const RAW_DIR           = joinpath(DATA_DIR, "raw")
const DERIVED_DIR       = joinpath(DATA_DIR, "derived")

const RAW_CPS_BASIC_DIR = joinpath(RAW_DIR, "cps_basic")
const RAW_CPS_ASEC_DIR  = joinpath(RAW_DIR, "cps_asec")
const RAW_JOLTS_DIR     = joinpath(RAW_DIR, "jolts")

mkpath(DERIVED_DIR)

println("PROJECT_ROOT = ", PROJECT_ROOT)
println("DERIVED_DIR  = ", DERIVED_DIR)

### Estimation Windows & Helper Functions

All the shared utilities from `utils.jl`.

In [ ]:
# ============================================================
# Estimation windows
# ============================================================

const WINDOWS = Dict{Symbol, NamedTuple}(
    :base_fc     => (label = "Pre-FC baseline",
                     ym_start = (2003, 1),  ym_end = (2007, 12),
                     asec_years = 2004:2008),
    :crisis_fc   => (label = "Financial crisis",
                     ym_start = (2008, 1),  ym_end = (2009, 6),
                     asec_years = 2009:2010),
    :base_covid  => (label = "Pre-COVID baseline",
                     ym_start = (2015, 1),  ym_end = (2019, 12),
                     asec_years = 2016:2020),
    :crisis_covid => (label = "COVID crisis",
                      ym_start = (2020, 3),  ym_end = (2021, 12),
                      asec_years = 2021:2022),
)

function in_window(year::Int, month::Int, win::NamedTuple)::Bool
    ym = (year, month)
    return ym >= win.ym_start && ym <= win.ym_end
end

function in_asec_window(survey_year::Int, win::NamedTuple)::Bool
    return survey_year in win.asec_years
end

# ============================================================
# Classification helpers
# ============================================================

is_skilled(educ::Integer)::Bool = educ >= 111
in_age_range(age::Integer; lo::Int=16, hi::Int=64)::Bool = lo <= age <= hi
is_civilian_lf(labforce::Integer)::Bool = labforce == 2
is_employed(empstat::Integer)::Bool = empstat in (10, 12)
is_unemployed(empstat::Integer)::Bool = empstat in (20, 21, 22)
is_wage_worker(classwkr::Integer)::Bool = classwkr in (22, 25, 27, 28)
is_enrolled_no_ba(schlcoll::Integer, educ::Integer)::Bool = schlcoll in (4, 5) && educ < 111
valid_match_mish(mish::Integer)::Bool = mish in (1, 2, 3, 5, 6, 7)

# ============================================================
# Wage construction
# ============================================================

function compute_hourly_wage(incwage, wkswork1, uhrsworkly)::Float64
    (wkswork1 <= 0 || uhrsworkly <= 0 || incwage <= 0) && return NaN
    return Float64(incwage) / (Float64(wkswork1) * Float64(uhrsworkly))
end

deflate_wage(nominal_wage::Float64, cpi_t::Float64, cpi_base::Float64)::Float64 =
    nominal_wage * (cpi_base / cpi_t)

# ============================================================
# Trimming
# ============================================================

function winsorize_bounds(wages::Vector{Float64}, weights::Vector{Float64};
                          lo_pct::Float64=0.01, hi_pct::Float64=0.99)
    n = length(wages)
    n == 0 && return (NaN, NaN)
    idx = sortperm(wages)
    cum = 0.0
    total = sum(weights)
    lo_val = wages[idx[1]]
    hi_val = wages[idx[end]]
    for i in idx
        cum += weights[i] / total
        if cum >= lo_pct && lo_val == wages[idx[1]]
            lo_val = wages[i]
        end
        if cum >= hi_pct
            hi_val = wages[i]
            break
        end
    end
    return (lo_val, hi_val)
end

# ============================================================
# Weighted statistics
# ============================================================

function wmean(x::AbstractVector, w::AbstractVector)::Float64
    sw = sum(w); sw <= 0.0 && return NaN
    return sum(x .* w) / sw
end

function wmedian(x::AbstractVector, w::AbstractVector)::Float64
    n = length(x); n == 0 && return NaN
    idx = sortperm(x)
    cum = 0.0; total = sum(w)
    total <= 0.0 && return NaN
    for i in idx
        cum += w[i] / total
        cum >= 0.5 && return Float64(x[i])
    end
    return Float64(x[idx[end]])
end

function wvar(x::AbstractVector, w::AbstractVector)::Float64
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^2) / sw
end

wsd(x::AbstractVector, w::AbstractVector)::Float64 = sqrt(max(wvar(x, w), 0.0))

function wcm3(x::AbstractVector, w::AbstractVector)::Float64
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^3) / sw
end

# ============================================================
# COVID correction (placeholder)
# ============================================================

function apply_covid_correction(empstat::Integer, year::Int, month::Int)
    (year == 2020 && 3 <= month <= 6) || return empstat
    return empstat
end

# ============================================================
# Moment name list
# ============================================================

const MOMENT_NAMES = [
    :ur_total, :ur_U, :ur_S, :skilled_share, :training_share,
    :emp_var_U, :emp_cm3_U, :emp_var_S, :emp_cm3_S,
    :jfr_U, :sep_rate_U, :jfr_S, :sep_rate_S,
    :ee_rate_S, :training_rate,
    :mean_wage_U, :mean_wage_S, :p50_wage_U, :p50_wage_S,
    :wage_premium, :wage_sd_U, :wage_sd_S,
    :theta_U, :theta_S,
]

println("All helpers loaded ✓")

### Check Raw Data Directories

In [ ]:
for (label, dir) in [("CPS Basic", RAW_CPS_BASIC_DIR),
                      ("CPS ASEC",  RAW_CPS_ASEC_DIR),
                      ("JOLTS",     RAW_JOLTS_DIR)]
    if isdir(dir)
        files = readdir(dir)
        @printf("  ✓ %-12s  %s  (%d files)\n", label, dir, length(files))
    else
        @printf("  ✗ %-12s  %s  (MISSING)\n", label, dir)
        mkpath(dir)
    end
end

---
## Stage 1 — Clean CPS Basic Monthly

Read raw IPUMS CPS Basic extract, restrict sample, classify skill / employment, apply COVID correction, assign windows, compute industry skill shares.

In [ ]:
function clean_cps_basic()
    @info "clean_cps_basic: reading raw data..."

    raw_files = readdir(RAW_CPS_BASIC_DIR)
    csv_file  = filter(f -> endswith(f, ".csv") || endswith(f, ".csv.gz"), raw_files)
    arrow_file = filter(f -> endswith(f, ".arrow"), raw_files)

    if !isempty(arrow_file)
        df = DataFrame(Arrow.Table(joinpath(RAW_CPS_BASIC_DIR, first(arrow_file))))
    elseif !isempty(csv_file)
        df = CSV.read(joinpath(RAW_CPS_BASIC_DIR, first(csv_file)), DataFrame)
    else
        error("No CSV or Arrow file found in $(RAW_CPS_BASIC_DIR)")
    end
    @info "  Raw records: $(nrow(df))"

    rename!(df, [Symbol(uppercase(string(c))) => c for c in names(df)]...)

    filter!(row -> in_age_range(row.AGE), df)
    filter!(row -> is_civilian_lf(row.LABFORCE), df)
    @info "  After age/LF restriction: $(nrow(df))"

    df.skilled     = is_skilled.(df.EDUC)
    df.employed    = is_employed.(df.EMPSTAT)
    df.unemployed  = is_unemployed.(df.EMPSTAT)

    # COVID BLS correction
    for i in 1:nrow(df)
        corrected = apply_covid_correction(df.EMPSTAT[i], df.YEAR[i], df.MONTH[i])
        if corrected != df.EMPSTAT[i]
            df.employed[i]   = is_employed(corrected)
            df.unemployed[i] = is_unemployed(corrected)
        end
    end

    # Training flag
    if hasproperty(df, :SCHLCOLL)
        df.in_training = is_enrolled_no_ba.(df.SCHLCOLL, df.EDUC)
    else
        @warn "SCHLCOLL not in data — setting in_training = false"
        df.in_training = fill(false, nrow(df))
    end

    df.valid_match = valid_match_mish.(df.MISH)

    # Window assignment
    df.window = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_window(df.YEAR[i], df.MONTH[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end
    @info "  Observations in estimation windows: $(count(w -> w != :none, df.window)) / $(nrow(df))"

    # Industry skill shares
    if hasproperty(df, :IND)
        emp_df = filter(row -> row.employed && row.window != :none, df)
        ind_shares = combine(
            groupby(emp_df, [:window, :IND]),
            :skilled => (s -> mean(s)) => :skilled_share_ind,
            :WTFINL  => sum => :weight_ind
        )
        Arrow.write(joinpath(DERIVED_DIR, "industry_skill_shares.arrow"), ind_shares)
        @info "  Industry skill shares saved ($(nrow(ind_shares)) rows)"
    end

    cols_to_keep = intersect(
        [:YEAR, :MONTH, :CPSID, :CPSIDP, :MISH, :WTFINL,
         :EMPSTAT, :EDUC, :AGE, :SEX, :IND,
         :skilled, :employed, :unemployed, :in_training,
         :valid_match, :window],
        Symbol.(names(df))
    )
    select!(df, cols_to_keep)

    outpath = joinpath(DERIVED_DIR, "cps_basic_clean.arrow")
    Arrow.write(outpath, df)
    @info "  Saved: $outpath  ($(nrow(df)) rows)"
    return df
end

cps_basic = clean_cps_basic()

### Explore CPS Basic

In [ ]:
println("Size: $(nrow(cps_basic)) rows × $(ncol(cps_basic)) cols")
println("Columns: ", names(cps_basic))
first(cps_basic, 30)

In [ ]:
describe(cps_basic)

In [ ]:
# Distribution across estimation windows
combine(groupby(cps_basic, :window), nrow => :count)

In [ ]:
# Skilled vs unskilled counts
combine(groupby(cps_basic, :skilled), nrow => :count)

---
## Stage 2 — Clean CPS ASEC

Read raw IPUMS CPS ASEC extract, restrict to wage workers, construct real hourly wages, trim, normalise.

In [ ]:
function clean_cps_asec()
    @info "clean_cps_asec: reading raw data..."

    raw_files = readdir(RAW_CPS_ASEC_DIR)
    csv_file   = filter(f -> endswith(f, ".csv") || endswith(f, ".csv.gz"), raw_files)
    arrow_file = filter(f -> endswith(f, ".arrow"), raw_files)

    if !isempty(arrow_file)
        df = DataFrame(Arrow.Table(joinpath(RAW_CPS_ASEC_DIR, first(arrow_file))))
    elseif !isempty(csv_file)
        df = CSV.read(joinpath(RAW_CPS_ASEC_DIR, first(csv_file)), DataFrame)
    else
        error("No CSV or Arrow file found in $(RAW_CPS_ASEC_DIR)")
    end
    @info "  Raw records: $(nrow(df))"

    rename!(df, [Symbol(uppercase(string(c))) => c for c in names(df)]...)

    filter!(row -> in_age_range(row.AGE), df)
    filter!(row -> is_wage_worker(row.CLASSWKR), df)
    filter!(row -> row.INCWAGE > 0 && row.WKSWORK1 > 0 && row.UHRSWORKLY > 0, df)
    @info "  After sample restrictions: $(nrow(df))"

    df.hourly_wage = compute_hourly_wage.(df.INCWAGE, df.WKSWORK1, df.UHRSWORKLY)
    filter!(row -> isfinite(row.hourly_wage) && row.hourly_wage > 0.0, df)

    # Deflate to constant (2012) dollars
    if hasproperty(df, :CPI99)
        cpi_2012 = df.CPI99[findfirst(==(2013), df.YEAR)]
        if isnothing(cpi_2012) || ismissing(cpi_2012)
            cpi_2012 = 1.0
            @warn "Could not identify CPI99 for 2012; wages not deflated."
        end
        df.real_wage = deflate_wage.(df.hourly_wage, Float64.(df.CPI99), Float64(cpi_2012))
    else
        @warn "CPI99 not in data — wages remain nominal."
        df.real_wage = df.hourly_wage
    end

    df.skilled = is_skilled.(df.EDUC)

    # Window assignment (ASEC survey year → income year)
    df.income_year = df.YEAR .- 1
    df.window = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_asec_window(df.YEAR[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end
    @info "  Observations in estimation windows: $(count(w -> w != :none, df.window)) / $(nrow(df))"

    # Trimming: 1st/99th percentile within each window
    df.trimmed = fill(false, nrow(df))
    for wname in keys(WINDOWS)
        mask = df.window .== wname
        !any(mask) && continue
        wages_w   = df.real_wage[mask]
        weights_w = Float64.(df.ASECWT[mask])
        lo, hi = winsorize_bounds(wages_w, weights_w)
        for i in findall(mask)
            if df.real_wage[i] < lo || df.real_wage[i] > hi
                df.trimmed[i] = true
            end
        end
    end
    @info "  Trimmed observations: $(count(df.trimmed))"
    filter!(row -> !row.trimmed, df)

    # Normalisation: divide by pooled median within each window
    df.wage_norm = fill(NaN, nrow(df))
    for wname in keys(WINDOWS)
        mask = df.window .== wname
        !any(mask) && continue
        wages_w   = df.real_wage[mask]
        weights_w = Float64.(df.ASECWT[mask])
        med_w     = wmedian(wages_w, weights_w)
        if isfinite(med_w) && med_w > 0.0
            df.wage_norm[mask] .= df.real_wage[mask] ./ med_w
        else
            @warn "Median wage is invalid for window $wname"
            df.wage_norm[mask] .= df.real_wage[mask]
        end
    end

    cols_to_keep = intersect(
        [:YEAR, :income_year, :ASECWT, :AGE, :SEX, :EDUC,
         :skilled, :real_wage, :wage_norm, :window],
        Symbol.(names(df))
    )
    select!(df, cols_to_keep)

    outpath = joinpath(DERIVED_DIR, "cps_asec_clean.arrow")
    Arrow.write(outpath, df)
    @info "  Saved: $outpath  ($(nrow(df)) rows)"
    return df
end

cps_asec = clean_cps_asec()

### Explore CPS ASEC

In [ ]:
println("Size: $(nrow(cps_asec)) rows × $(ncol(cps_asec)) cols")
first(cps_asec, 30)

In [ ]:
describe(cps_asec)

In [ ]:
# Wage stats by skill group
combine(groupby(cps_asec, :skilled),
    :real_wage => mean => :mean_real_wage,
    :wage_norm => mean => :mean_wage_norm,
    :real_wage => std  => :sd_real_wage,
    nrow => :count
)

In [ ]:
# Wage stats by window
combine(groupby(filter(r -> r.window != :none, cps_asec), :window),
    :real_wage => mean => :mean_real_wage,
    :wage_norm => mean => :mean_wage_norm,
    nrow => :count
)

---
## Stage 3 — Clean JOLTS

Read BLS JOLTS flat files, parse series IDs, assign windows, merge with CPS industry skill shares, allocate vacancies to skill segments.

In [ ]:
function _parse_jolts_industry(sid::String)::String
    m = match(r"^JTS(\d{6,})\d*JOL$", sid)
    isnothing(m) && return "unknown"
    return m.captures[1]
end

function clean_jolts()
    @info "clean_jolts: reading raw data..."

    raw_files = readdir(RAW_JOLTS_DIR)
    csv_files = filter(f -> endswith(f, ".csv") || endswith(f, ".csv.gz"), raw_files)
    isempty(csv_files) && error("No CSV files found in $(RAW_JOLTS_DIR)")

    dfs = DataFrame[]
    for f in csv_files
        push!(dfs, CSV.read(joinpath(RAW_JOLTS_DIR, f), DataFrame))
    end
    df = vcat(dfs...; cols=:union)
    @info "  Raw records: $(nrow(df))"

    rename!(df, [Symbol(uppercase(string(c))) => c for c in names(df)]...)

    if hasproperty(df, :SERIESID) && !hasproperty(df, :SERIES_ID)
        rename!(df, :SERIESID => :SERIES_ID)
    end

    if hasproperty(df, :PERIOD)
        df.MONTH = [parse(Int, replace(string(p), r"^M0?" => "")) for p in df.PERIOD]
        filter!(row -> 1 <= row.MONTH <= 12, df)
    end

    df.industry_code = [_parse_jolts_industry(string(s)) for s in df.SERIES_ID]
    filter!(row -> endswith(string(row.SERIES_ID), "JOL"), df)

    df.window = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_window(df.YEAR[i], df.MONTH[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end

    # Merge with CPS industry skill shares
    shares_path = joinpath(DERIVED_DIR, "industry_skill_shares.arrow")
    if isfile(shares_path)
        shares = DataFrame(Arrow.Table(shares_path))
        @info "  Loaded industry skill shares ($(nrow(shares)) rows)"
        df_merged = leftjoin(df, shares;
                             on = [:window => :window, :industry_code => :IND],
                             makeunique = true)
        for i in 1:nrow(df_merged)
            if ismissing(df_merged.skilled_share_ind[i])
                df_merged.skilled_share_ind[i] = 0.5
            end
        end
        df_merged.V_S = df_merged.VALUE .* df_merged.skilled_share_ind
        df_merged.V_U = df_merged.VALUE .* (1.0 .- df_merged.skilled_share_ind)
        df = df_merged
    else
        @warn "Industry skill shares not found — using equal split for V_S/V_U"
        df.V_S = df.VALUE .* 0.5
        df.V_U = df.VALUE .* 0.5
    end

    by_industry = filter(row -> row.industry_code != "000000", df)
    if nrow(by_industry) > 0
        agg = combine(
            groupby(by_industry, [:YEAR, :MONTH, :window]),
            :V_S => sum => :V_S,
            :V_U => sum => :V_U,
        )
    else
        total_rows = filter(row -> row.industry_code == "000000", df)
        agg = combine(
            groupby(total_rows, [:YEAR, :MONTH, :window]),
            :V_S => sum => :V_S,
            :V_U => sum => :V_U,
        )
    end

    outpath = joinpath(DERIVED_DIR, "jolts_clean.arrow")
    Arrow.write(outpath, agg)
    @info "  Saved: $outpath  ($(nrow(agg)) rows)"
    return agg
end

jolts = clean_jolts()

### Explore JOLTS

In [ ]:
println("Size: $(nrow(jolts)) rows × $(ncol(jolts)) cols")
first(jolts, 30)

In [ ]:
describe(jolts)

In [ ]:
# Vacancies by window
combine(groupby(filter(r -> r.window != :none, jolts), :window),
    :V_S => mean => :mean_V_S,
    :V_U => mean => :mean_V_U,
    nrow => :n_months
)

---
## Stage 4 — Build Transition Rates

Match persons across adjacent CPS months using CPSIDP, compute job-finding and separation rates by skill group.

In [ ]:
function make_transitions()
    @info "make_transitions: loading cleaned CPS Basic..."

    inpath = joinpath(DERIVED_DIR, "cps_basic_clean.arrow")
    !isfile(inpath) && error("cps_basic_clean.arrow not found — run Stage 1 first")
    df = DataFrame(Arrow.Table(inpath))

    @info "  Building matched month-pairs..."
    matchable = filter(row -> row.valid_match && row.CPSIDP > 0, df)
    next_ym(y, m) = m == 12 ? (y + 1, 1) : (y, m + 1)

    lookup = Dict{Tuple{Int64, Int, Int}, NamedTuple}()
    for row in eachrow(df)
        key = (Int64(row.CPSIDP), row.YEAR, row.MONTH)
        lookup[key] = (skilled=row.skilled, employed=row.employed,
                       unemployed=row.unemployed, weight=row.WTFINL,
                       window=row.window)
    end

    pairs_data = NamedTuple[]
    for row in eachrow(matchable)
        ny, nm = next_ym(row.YEAR, row.MONTH)
        next_key = (Int64(row.CPSIDP), ny, nm)
        haskey(lookup, next_key) || continue
        next = lookup[next_key]
        push!(pairs_data, (
            year_t=row.YEAR, month_t=row.MONTH, skilled=row.skilled,
            emp_t=row.employed, unemp_t=row.unemployed,
            emp_t1=next.employed, unemp_t1=next.unemployed,
            weight=Float64(row.WTFINL), window=row.window,
        ))
    end
    pairs = DataFrame(pairs_data)
    @info "  Matched pairs: $(nrow(pairs))"

    # Monthly transition rates
    results = NamedTuple[]
    for gk in groupby(pairs, [:year_t, :month_t, :skilled, :window])
        g = DataFrame(gk)
        sk = g.skilled[1]; win = g.window[1]
        u_mask  = g.unemp_t
        ue_mask = g.unemp_t .& g.emp_t1
        denom_jfr = sum(g.weight[u_mask])
        numer_jfr = sum(g.weight[ue_mask])
        jfr = denom_jfr > 0 ? numer_jfr / denom_jfr : NaN
        e_mask  = g.emp_t
        eu_mask = g.emp_t .& g.unemp_t1
        denom_sep = sum(g.weight[e_mask])
        numer_sep = sum(g.weight[eu_mask])
        sep = denom_sep > 0 ? numer_sep / denom_sep : NaN
        push!(results, (year=g.year_t[1], month=g.month_t[1], skilled=sk,
                         window=win, jfr=jfr, sep=sep, n_pairs=nrow(g)))
    end
    rates = DataFrame(results)

    # Window averages
    window_rates = NamedTuple[]
    for gk in groupby(filter(r -> r.window != :none, rates), [:window, :skilled])
        g = DataFrame(gk)
        valid_jfr = filter(isfinite, g.jfr)
        valid_sep = filter(isfinite, g.sep)
        push!(window_rates, (
            window=g.window[1], skilled=g.skilled[1],
            mean_jfr=isempty(valid_jfr) ? NaN : mean(valid_jfr),
            mean_sep=isempty(valid_sep) ? NaN : mean(valid_sep),
            n_months=nrow(g),
        ))
    end
    agg = DataFrame(window_rates)

    Arrow.write(joinpath(DERIVED_DIR, "transitions_monthly.arrow"), rates)
    outpath = joinpath(DERIVED_DIR, "transitions.arrow")
    Arrow.write(outpath, agg)
    @info "  Saved: $outpath  ($(nrow(agg)) rows)"
    return agg
end

transitions = make_transitions()

### Explore Transitions

In [ ]:
println("Window-averaged transition rates:")
transitions

In [ ]:
# Also look at monthly rates
rates_monthly = DataFrame(Arrow.Table(joinpath(DERIVED_DIR, "transitions_monthly.arrow")))
println("Monthly rates: $(nrow(rates_monthly)) rows")
first(rates_monthly, 30)

In [ ]:
describe(rates_monthly)

---
## Stage 5 — Compute All Moments

24 targeted moments × 4 estimation windows.

In [ ]:
function _load_arrow(filename::String)::DataFrame
    path = joinpath(DERIVED_DIR, filename)
    !isfile(path) && (@warn "$filename not found"; return DataFrame())
    return DataFrame(Arrow.Table(path))
end

function _compute_stock_moments(cps_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    monthly = NamedTuple[]
    for gk in groupby(cps_w, [:YEAR, :MONTH])
        g = DataFrame(gk)
        w = Float64.(g.WTFINL); sw = sum(w)
        sw <= 0 && continue
        n_unemp   = sum(w[g.unemployed])
        n_unemp_U = sum(w[g.unemployed .& .!g.skilled])
        n_unemp_S = sum(w[g.unemployed .& g.skilled])
        n_lf = sw; n_lf_U = sum(w[.!g.skilled]); n_lf_S = sum(w[g.skilled])
        ur_total = n_lf > 0 ? n_unemp / n_lf : NaN
        ur_U = n_lf_U > 0 ? n_unemp_U / n_lf_U : NaN
        ur_S = n_lf_S > 0 ? n_unemp_S / n_lf_S : NaN
        skilled_share = n_lf > 0 ? n_lf_S / n_lf : NaN
        training_share = hasproperty(g, :in_training) ? (n_lf > 0 ? sum(w[g.in_training]) / n_lf : NaN) : NaN
        push!(monthly, (ur_total=ur_total, ur_U=ur_U, ur_S=ur_S,
                         skilled_share=skilled_share, training_share=training_share))
    end
    if !isempty(monthly)
        mdf = DataFrame(monthly)
        moments[:ur_total]       = mean(filter(isfinite, mdf.ur_total))
        moments[:ur_U]           = mean(filter(isfinite, mdf.ur_U))
        moments[:ur_S]           = mean(filter(isfinite, mdf.ur_S))
        moments[:skilled_share]  = mean(filter(isfinite, mdf.skilled_share))
        moments[:training_share] = mean(filter(isfinite, mdf.training_share))
    end
    return moments
end

function _fill_transition_moments!(moments::Dict{Symbol, Float64}, trans_w::DataFrame)
    for row in eachrow(trans_w)
        if row.skilled
            moments[:jfr_S]      = row.mean_jfr
            moments[:sep_rate_S] = row.mean_sep
        else
            moments[:jfr_U]      = row.mean_jfr
            moments[:sep_rate_U] = row.mean_sep
        end
    end
    moments[:ee_rate_S] = get(moments, :ee_rate_S, NaN)
end

function _compute_wage_moments(asec_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    unskilled = filter(row -> !row.skilled, asec_w)
    skilled   = filter(row ->  row.skilled, asec_w)
    if nrow(unskilled) > 0
        wu = Float64.(unskilled.wage_norm); wt = Float64.(unskilled.ASECWT)
        moments[:mean_wage_U] = wmean(wu, wt); moments[:p50_wage_U] = wmedian(wu, wt)
        moments[:wage_sd_U] = wsd(wu, wt); moments[:emp_var_U] = wvar(wu, wt)
        moments[:emp_cm3_U] = wcm3(wu, wt)
    end
    if nrow(skilled) > 0
        ws = Float64.(skilled.wage_norm); wt = Float64.(skilled.ASECWT)
        moments[:mean_wage_S] = wmean(ws, wt); moments[:p50_wage_S] = wmedian(ws, wt)
        moments[:wage_sd_S] = wsd(ws, wt); moments[:emp_var_S] = wvar(ws, wt)
        moments[:emp_cm3_S] = wcm3(ws, wt)
    end
    if nrow(unskilled) > 0 && nrow(skilled) > 0
        log_wu = log.(max.(Float64.(unskilled.wage_norm), 1e-14))
        log_ws = log.(max.(Float64.(skilled.wage_norm), 1e-14))
        moments[:wage_premium] = wmean(log_ws, Float64.(skilled.ASECWT)) - wmean(log_wu, Float64.(unskilled.ASECWT))
    end
    return moments
end

function _compute_tightness(jolts_w::DataFrame, cps_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    mean_V_U = mean(jolts_w.V_U); mean_V_S = mean(jolts_w.V_S)
    monthly_U = NamedTuple[]
    for gk in groupby(cps_w, [:YEAR, :MONTH])
        g = DataFrame(gk); w = Float64.(g.WTFINL)
        push!(monthly_U, (U_U=sum(w[g.unemployed .& .!g.skilled]),
                           U_S=sum(w[g.unemployed .& g.skilled])))
    end
    mdf = DataFrame(monthly_U)
    mean_U_U = mean(mdf.U_U); mean_U_S = mean(mdf.U_S)
    moments[:theta_U] = mean_U_U > 0 ? mean_V_U / mean_U_U : NaN
    moments[:theta_S] = mean_U_S > 0 ? mean_V_S / mean_U_S : NaN
    return moments
end

function make_moments()
    @info "make_moments: assembling all 24 moments × 4 windows..."
    cps_basic_m = _load_arrow("cps_basic_clean.arrow")
    cps_asec_m  = _load_arrow("cps_asec_clean.arrow")
    trans       = _load_arrow("transitions.arrow")
    jolts_m     = _load_arrow("jolts_clean.arrow")

    all_moments = Dict{Symbol, DataFrame}()
    for (wname, wdef) in WINDOWS
        @info "  Window: $(wdef.label) ($wname)"
        moments = Dict{Symbol, Float64}()
        cps_w = filter(row -> row.window == wname, cps_basic_m)
        nrow(cps_w) > 0 ? merge!(moments, _compute_stock_moments(cps_w)) :
            (for k in [:ur_total,:ur_U,:ur_S,:skilled_share,:training_share]; moments[k]=NaN; end)
        trans_w = filter(row -> row.window == wname, trans)
        nrow(trans_w) > 0 ? _fill_transition_moments!(moments, trans_w) :
            (for k in [:jfr_U,:sep_rate_U,:jfr_S,:sep_rate_S,:ee_rate_S,:training_rate]; moments[k]=NaN; end)
        if haskey(moments, :training_share) && haskey(moments, :ur_U)
            ts = moments[:training_share]; ur = moments[:ur_U]
            moments[:training_rate] = (ts+ur) > 0 ? ts/(ts+ur) : NaN
        end
        asec_w = filter(row -> row.window == wname, cps_asec_m)
        nrow(asec_w) > 0 ? merge!(moments, _compute_wage_moments(asec_w)) :
            (for k in [:mean_wage_U,:mean_wage_S,:p50_wage_U,:p50_wage_S,:wage_premium,:wage_sd_U,:wage_sd_S,:emp_var_U,:emp_cm3_U,:emp_var_S,:emp_cm3_S]; moments[k]=NaN; end)
        jolts_w = filter(row -> row.window == wname, jolts_m)
        (nrow(jolts_w) > 0 && nrow(cps_w) > 0) ? merge!(moments, _compute_tightness(jolts_w, cps_w)) :
            (moments[:theta_U]=NaN; moments[:theta_S]=NaN)

        moment_df = DataFrame(moment=String[], value=Float64[], se=Float64[], weight=Float64[])
        for mname in MOMENT_NAMES
            val = get(moments, mname, NaN)
            se = isfinite(val) && val != 0.0 ? abs(val)*0.10 : 1.0
            wt = isfinite(se) && se > 0.0 ? 1.0/se^2 : 0.0
            push!(moment_df, (string(mname), val, se, wt))
        end
        CSV.write(joinpath(DERIVED_DIR, "moments_$(wname).csv"), moment_df)

        K = length(MOMENT_NAMES)
        sigma = zeros(K, K)
        for (i, _) in enumerate(MOMENT_NAMES); sigma[i,i] = moment_df.se[i]^2; end
        sigma_df = DataFrame(sigma, [string(m) for m in MOMENT_NAMES])
        CSV.write(joinpath(DERIVED_DIR, "sigma_$(wname).csv"), sigma_df)

        all_moments[wname] = moment_df
    end
    @info "  All moment files saved to $(DERIVED_DIR)"
    return all_moments
end

all_moments = make_moments()

### Explore Moments

In [ ]:
# Print all moments for each window
for (wname, mdf) in all_moments
    println("\n═══ Window: $wname ═══")
    display(mdf)
end

In [ ]:
# Quick comparison: pick two windows side by side
if haskey(all_moments, :base_fc) && haskey(all_moments, :crisis_fc)
    comparison = DataFrame(
        moment = all_moments[:base_fc].moment,
        base_fc = all_moments[:base_fc].value,
        crisis_fc = all_moments[:crisis_fc].value,
    )
    comparison.diff = comparison.crisis_fc .- comparison.base_fc
    comparison
end

---
## Pipeline Summary

List all derived files produced by the pipeline.

In [ ]:
println("Derived files in: $DERIVED_DIR")
for f in sort(readdir(DERIVED_DIR))
    sz = filesize(joinpath(DERIVED_DIR, f))
    @printf("  %-40s  %s\n", f, sz < 1024 ? "$sz B" : sz < 1024^2 ? "$(round(sz/1024; digits=1)) KB" : "$(round(sz/1024^2; digits=1)) MB")
end